# Preprocesamiento de Datos
En este notebook se limpia y prepara el dataset para alimentar el modelo generativo.
Las transformaciones aplicadas incluyen selección de columnas relevantes, 
manejo de valores nulos y creación de variables derivadas útiles para el prompting.

In [1]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/Matches.csv")
df_epl = df_raw[df_raw['Division'] == 'E0'].copy()
df_epl['MatchDate'] = pd.to_datetime(df_epl['MatchDate'])
print(df_epl.shape)

(9410, 48)


C:\Users\benja\AppData\Local\Temp\ipykernel_28804\1619480576.py:3: DtypeWarning: Columns (0: MatchTime) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("../data/raw/Matches.csv")


## Selección de columnas relevantes para la generación de crónicas

In [2]:
cols = [
    'MatchDate', 'HomeTeam', 'AwayTeam',
    'FTHome', 'FTAway', 'FTResult',
    'HTHome', 'HTAway', 'HTResult',
    'HomeShots', 'AwayShots',
    'HomeTarget', 'AwayTarget',
    'HomeFouls', 'AwayFouls',
    'HomeCorners', 'AwayCorners',
    'HomeYellow', 'AwayYellow',
    'HomeRed', 'AwayRed',
    'Form5Home', 'Form5Away',
    'OddHome', 'OddDraw', 'OddAway',
    'HomeElo', 'AwayElo'
]

df = df_epl[cols].copy()
print(df.shape)

(9410, 28)


## Manejo de valores nulos
Se eliminan los partidos sin cuotas (83 registros) ya que son temporadas muy antiguas.
El resto de columnas críticas no tienen nulos.

In [3]:
df = df.dropna(subset=['OddHome', 'OddDraw', 'OddAway'])
df = df.dropna(subset=['HomeElo', 'AwayElo'])
print(f"Partidos tras limpieza: {df.shape[0]}")

Partidos tras limpieza: 9325


## Variables derivadas
Se crean columnas adicionales que enriquecen el contexto narrativo para el modelo generativo.

In [5]:
# Diferencia de goles
df['GoalDiff'] = df['FTHome'] - df['FTAway']

# Favorito según cuotas
df['Favorite'] = df.apply(
    lambda r: r['HomeTeam'] if r['OddHome'] < r['OddAway'] 
              else (r['AwayTeam'] if r['OddAway'] < r['OddHome'] else 'Ninguno'), axis=1
)

# Si ganó el favorito o hubo sorpresa
df['Upset'] = df.apply(
    lambda r: (r['FTResult'] == 'H' and r['OddHome'] > r['OddAway']) or
              (r['FTResult'] == 'A' and r['OddAway'] > r['OddHome']), axis=1
)

# Diferencia de Elo (superioridad relativa)
df['EloDiff'] = (df['HomeElo'] - df['AwayElo']).round(1)

# Precisión de tiros (evitar división por cero)
df['HomePrecision'] = (df['HomeTarget'] / df['HomeShots'].replace(0, float('nan'))).round(2)
df['AwayPrecision'] = (df['AwayTarget'] / df['AwayShots'].replace(0, float('nan'))).round(2)

df.head(3)

,MatchDate,HomeTeam,AwayTeam,FTHome,FTAway,FTResult,HTHome,HTAway,HTResult,HomeShots,...,OddDraw,OddAway,HomeElo,AwayElo,GoalDiff,Favorite,Upset,EloDiff,HomePrecision,AwayPrecision
154,2000-08-19,Charlton,Man City,4.0,0.0,H,2.0,0.0,H,17.0,...,3.0,3.2,1608.77,1579.99,4.0,Charlton,False,28.8,0.82,0.50
155,2000-08-19,Chelsea,West Ham,4.0,2.0,H,1.0,0.0,H,17.0,...,3.4,5.2,1800.17,1681.36,2.0,Chelsea,False,118.8,0.59,0.42
156,2000-08-19,Coventry,Middlesbrough,1.0,3.0,A,1.0,1.0,D,6.0,...,3.0,3.0,1635.61,1679.18,-2.0,Coventry,True,-43.6,0.50,0.56


## Exportación del dataset procesado

In [7]:
df.to_csv("../data/processed/epl_clean.csv", index=False)
print(f"Dataset guardado: {df.shape[0]} partidos, {df.shape[1]} columnas")

Dataset guardado: 9325 partidos, 34 columnas
